<a href="https://colab.research.google.com/github/csnorbi18/Projekt/blob/master/BPI_Challenge_2017.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.0 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import pandas as pd
import pm4py
import gzip
import shutil
import os



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Feltöltött fájl:", filename)

Saving BPI Challenge 2017.xes.gz to BPI Challenge 2017.xes.gz
Feltöltött fájl: BPI Challenge 2017.xes.gz


In [10]:
if not Path(filename).is_file():
    raise FileNotFoundError(f"Nem található a fájl: {filename}")

# XES és tömörített XES fájlok kezelése
if filename.lower().endswith((".xes", ".xes.gz")):

    # A PM4Py közvetlenül támogatja a .xes.gz fájlokat
    log = pm4py.read_xes(filename)

    # Átalakítás DataFrame formátumra, ha szükséges
    if not isinstance(log, pd.DataFrame):
        log = pm4py.convert_to_dataframe(log)

    # Kötelező PM4Py-oszlopok
    required = [
        "case:concept:name",
        "concept:name",
        "time:timestamp"
    ]

    missing = [c for c in required if c not in log.columns]

    if missing:
        raise ValueError(
            "A XES fájlból hiányzó kötelező attribútum(ok): "
            + ", ".join(missing)
        )

    # Időbélyegek egységesítése
    log["time:timestamp"] = pd.to_datetime(
        log["time:timestamp"],
        utc=True
    )

    # Egységes PM4Py DataFrame-formátum
    log = pm4py.format_dataframe(
        log,
        case_id="case:concept:name",
        activity_key="concept:name",
        timestamp_key="time:timestamp"
    )

    print("XES eseménynapló sikeresen betöltve.")

else:
    raise ValueError("Csak .xes vagy .xes.gz fájl támogatott.")

# Rendezés esetszám és időbélyeg alapján
log = (
    log.sort_values(
        ["case:concept:name", "time:timestamp"],
        kind="stable"
    )
    .reset_index(drop=True)
)

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

XES eseménynapló sikeresen betöltve.
Események száma: 1,202,267
Esetek száma: 31,509
Tevékenységek száma: 26


,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,...,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,@@index,@@case_index
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,2016-08-03 15:57:21.673000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,2016-08-03 15:57:21.734000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,2016-08-03 15:57:21.963000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1030261128,withdraw,2016-08-03 15:58:28.286000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
4,Created,User_1,W_Complete application,Workflow,Workitem_1127124826,schedule,2016-08-03 15:58:28.293000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0


In [15]:
n_events = len(log)
n_cases = log["case:concept:name"].nunique()
n_activities = log["concept:name"].nunique()

print("Események száma:", n_events)
print("Esetek száma:", n_cases)
print("Tevékenységek száma:", n_activities)
print("Átlagos eseményszám esetenként:", round(n_events / n_cases, 2) if n_cases else None)

print("\nOszlopok:")
print(list(log.columns))
print("\nAdattípusok:")
print(log.dtypes)

log.head(3)

Események száma: 1202267
Esetek száma: 31509
Tevékenységek száma: 26
Átlagos eseményszám esetenként: 38.16

Oszlopok:
['Action', 'org:resource', 'concept:name', 'EventOrigin', 'EventID', 'lifecycle:transition', 'time:timestamp', 'case:LoanGoal', 'case:ApplicationType', 'case:concept:name', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount', 'OfferID', '@@index', '@@case_index']

Adattípusok:
Action                                object
org:resource                          object
concept:name                  string[python]
EventOrigin                           object
EventID                               object
lifecycle:transition                  object
time:timestamp           datetime64[ns, UTC]
case:LoanGoal                         object
case:ApplicationType                  object
case:concept:name             string[python]
case:RequestedAmount                 float64
FirstWithdrawalAmount    

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,...,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,@@index,@@case_index
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,2016-08-03 15:57:21.673000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,2016-08-03 15:57:21.734000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,2016-08-03 15:57:21.963000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
